# Zero-Day Attack Classification — CICIoT2023 Pilot

**Experiment:** Withhold one attack class entirely from training. Generate binary (normal/attack) rules using the RAG-LLM pipeline on the remaining data. Evaluate on the withheld class at test time to measure zero-day generalisation.

**Key metric:** Zero-Day Detection Rate (ZDR) = fraction of withheld-class entries flagged as 'attack'.

**To run a different withheld class:** change `WITHHELD_CLASS` in Cell 1 and re-run all cells.

In [ ]:
################################################################################
# Cell 0 — Configuration
################################################################################

# === Change this to run a different zero-day scenario ===
#
# CICIoT2023 recommended candidates (in order of scientific interest):
#   'Backdoor_Malware'  — stealth/persistence vs. volumetric floods (RECOMMENDED)
#   'SqlInjection'      — application-layer vs. network-layer floods
#   'MITM-ArpSpoofing'  — ARP-layer, ARP binary indicator feature
#   'XSS'               — web-layer cross-site scripting
#   'CommandInjection'  — OS command injection

WITHHELD_CLASS = 'Backdoor_Malware'

DATASET_NAME = 'cic-iot'
N_NORMAL     = 10800   # benign samples in train pool (matches original binary experiment)
N_ATTACK     = 89200   # known-attack samples in train pool
N_REPR       = 10      # representative samples per class fed to LLM
k            = 5       # number of rules
n            = 5       # number of feedback iterations
SEEDS        = [42, 123, 456]  # repeated runs for error bars
SEED_MAIN    = 42      # primary seed for this run

print(f'Zero-day class: {WITHHELD_CLASS}')
print(f'Rules: {k}, Iterations: {n}, Seeds: {SEEDS}')

In [ ]:
################################################################################
# Cell 1 — Load multiclass population (original labels intact)
################################################################################

import pandas as pd
import numpy as np
import os
from tabulate import tabulate

df = pd.read_csv(os.getcwd() + '/data/population.csv')

print(f'Total records: {len(df):,}')
print(f'\nClass distribution:')
class_counts = df['label'].value_counts()
print(class_counts.to_string())
print(f'\nTotal classes: {df["label"].nunique()}')
print(f'Withheld class "{WITHHELD_CLASS}" count: {class_counts.get(WITHHELD_CLASS, 0):,}')

In [ ]:
################################################################################
# Cell 2 — Prepare zero-day data split
################################################################################

# Partition into three groups
benign_df    = df[df['label'] == 'BenignTraffic'].copy()
known_atk_df = df[(df['label'] != 'BenignTraffic') & (df['label'] != WITHHELD_CLASS)].copy()
zeroday_df   = df[df['label'] == WITHHELD_CLASS].copy()

print(f'Benign:          {len(benign_df):>8,}')
print(f'Known attacks:   {len(known_atk_df):>8,}  ({known_atk_df["label"].nunique()} classes)')
print(f'Zero-day class:  {len(zeroday_df):>8,}  ← "{WITHHELD_CLASS}" (withheld)')

# Build binary training pool (no zero-day data)
n_attack_avail = min(N_ATTACK, len(known_atk_df))
if n_attack_avail < N_ATTACK:
    print(f'\nWARNING: Only {n_attack_avail:,} known-attack samples available (< {N_ATTACK:,}). Using all.')

train_pool = pd.concat([
    benign_df.sample(n=min(N_NORMAL, len(benign_df)), random_state=SEED_MAIN),
    known_atk_df.sample(n=n_attack_avail, random_state=SEED_MAIN)
], ignore_index=True)

train_pool['label'] = train_pool['label'].apply(
    lambda x: 'normal' if x == 'BenignTraffic' else 'attack'
)

# 80/20 split on the known-data pool
train_df      = train_pool.sample(frac=0.8, random_state=SEED_MAIN)
test_known_df = train_pool.drop(train_df.index)

# Separate feature and label columns
feature_cols = [c for c in train_df.columns if c != 'label']

normal_df_train  = train_df[train_df['label'] == 'normal'][feature_cols].reset_index(drop=True)
attack_df_train  = train_df[train_df['label'] == 'attack'][feature_cols].reset_index(drop=True)
normal_df_test   = test_known_df[test_known_df['label'] == 'normal'][feature_cols].reset_index(drop=True)
attack_df_test   = test_known_df[test_known_df['label'] == 'attack'][feature_cols].reset_index(drop=True)
zeroday_df_test  = zeroday_df[feature_cols].reset_index(drop=True)  # ALL zero-day samples

data = [
    ['Normal (train)',         len(normal_df_train)],
    ['Known attack (train)',   len(attack_df_train)],
    ['Normal (test)',          len(normal_df_test)],
    ['Known attack (test)',    len(attack_df_test)],
    [f'Zero-day test ({WITHHELD_CLASS})', len(zeroday_df_test)],
]
print('\nDataset splits:')
print(tabulate(data, headers=['Split', 'Count'], tablefmt='grid'))
print(f'\nFeatures: {len(feature_cols)}')
print(f'Note: Zero-day class was NOT included in any training split.')

In [ ]:
################################################################################
# Cell 3 — Get representative samples for LLM prompt
#
# We use feature-space nearest-to-mean selection (no Chroma rebuild required).
# This is functionally equivalent to the original RAG retrieval for the purpose
# of generating LLM prompts, and avoids rebuilding a 100k-embedding vector store.
#
# To use full Chroma embeddings (exact replication of original methodology),
# build a new Chroma collection from train_df and persist to
# ./vector-stores/chroma-db-zeroday-{WITHHELD_CLASS}/
################################################################################

import json
from scipy.spatial.distance import cdist


def get_representative_samples(df: pd.DataFrame, n: int = 10) -> pd.DataFrame:
    """Return n rows closest (Euclidean) to the column-wise mean — the most 'typical' entries."""
    mean_vec = df.mean(axis=0).values.reshape(1, -1)
    distances = cdist(df.values, mean_vec, metric='euclidean').flatten()
    indices = np.argsort(distances)[:n]
    return df.iloc[indices]


normal_repr  = get_representative_samples(normal_df_train, n=N_REPR)
attack_repr  = get_representative_samples(attack_df_train, n=N_REPR)

# Build the dicts that go into the LLM prompt (same format as original pipeline)
normal_entries_dict = {col: normal_repr[col].tolist() for col in feature_cols}
attack_entries_dict = {col: attack_repr[col].tolist() for col in feature_cols}

normal_entries = json.dumps(normal_entries_dict)
attack_entries = json.dumps(attack_entries_dict)

print(f'Representative normal  samples: {len(normal_repr)}')
print(f'Representative attack  samples: {len(attack_repr)}')
print(f'\nSample normal entry (first feature values):')
print({k: v[0] for k, v in list(normal_entries_dict.items())[:5]})

In [ ]:
################################################################################
# Cell 4 — Policy Evaluation Tool (identical to original pipeline)
################################################################################

import operator as op_module
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix
from statistics import mode
from typing import Annotated
from langchain_core.tools import tool

show_progress = False
operators = {
    '<':  op_module.lt,
    '>':  op_module.gt,
    '==': op_module.eq,
    '<=': op_module.le,
    '>=': op_module.ge,
    '!=': op_module.ne,
}


@tool
def evaluation_tool(
        feature_name: Annotated[str, 'Feature name'],
        value: Annotated[float, 'Threshold value'],
        op: Annotated[str, 'Operator (<, >, <=, >=, ==, !=)']
) -> float:
    """Evaluate a single threshold rule on training data. Returns macro F1-score."""
    try:
        value = float(value)
    except (ValueError, TypeError):
        pass
    if op not in operators:
        raise ValueError(f'Unsupported operator: {op}')
    datasets = {'normal': normal_df_train, 'attack': attack_df_train}
    y_pred, y_true = [], []
    for label, dataset in datasets.items():
        for i in tqdm(range(len(dataset)), disable=not show_progress,
                      ncols=100, desc=f'Evaluating {label}...'):
            y_true.append(label)
            try:
                y_pred.append('attack' if operators[op](dataset.iloc[i][feature_name], value) else 'normal')
            except (KeyError, TypeError):
                y_pred.append('normal')
    report = classification_report(y_true, y_pred, digits=4, output_dict=True)
    return report['macro avg']['f1-score']


print('evaluation_tool defined.')

In [ ]:
################################################################################
# Cell 5 — LangGraph Pipeline (identical to original pipeline)
################################################################################

import dotenv, os, uuid
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import MessagesState, StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver
from IPython.display import Image, display

dotenv.load_dotenv(os.getcwd() + '/../.env')


# ── State ──────────────────────────────────────────────────────────────────────
class State(MessagesState):
    i: int
    max_f1s: float
    best_tool_calls: list


# ── LLM ────────────────────────────────────────────────────────────────────────
llm = ChatOpenAI(model='gpt-4o', temperature=0.1)
# llm = ChatGoogleGenerativeAI(model='gemini-1.5-pro', temperature=0.1)  # alternative
tools = [evaluation_tool]
llm_with_tools = llm.bind_tools(tools)


# ── Nodes ──────────────────────────────────────────────────────────────────────
def llm_node(state):
    completion = llm_with_tools.invoke(state['messages'])
    return {'messages': [completion], 'i': state['i'] + 1}


def evaluate_node(state):
    """Evaluate current rules on test set (known attacks only), send feedback."""
    ai_messages = [m for m in state['messages'] if isinstance(m, AIMessage) and m.tool_calls]
    if not ai_messages:
        return {}
    tool_calls = ai_messages[-1].tool_calls

    datasets = {'normal': normal_df_test, 'attack': attack_df_test}
    y_pred, y_true = [], []
    for label, dataset in datasets.items():
        for i in tqdm(range(len(dataset)), disable=False, ncols=100, desc=f'Test eval {label}...'):
            try:
                votes = [
                    'attack' if operators[tc['args']['op']](
                        dataset.iloc[i][tc['args']['feature_name']], float(tc['args']['value'])
                    ) else 'normal'
                    for tc in tool_calls
                ]
                y_pred.append(mode(votes))
            except Exception:
                y_pred.append('normal')
            y_true.append(label)

    report  = classification_report(y_true, y_pred, digits=4, output_dict=True)
    matrix  = confusion_matrix(y_true, y_pred)
    f1      = report['macro avg']['f1-score']
    print(f'  Iter {state["i"]}: test macro-F1 = {f1:.4f}')
    print(matrix)

    # Track best rules
    new_max = max(state['max_f1s'], f1)
    new_best = tool_calls if f1 >= state['max_f1s'] else state['best_tool_calls']

    feedback = HumanMessage(
        f'Current macro avg F1 on test set: {f1:.4f}. '
        f'Best so far: {new_max:.4f}. '
        f'If this score is greater than the previous best, keep performing rules and revise the rest. '
        f'Otherwise revise all rules to exceed the best. '
        f'Generate exactly {k} rules and make a tool call for each.'
    )
    return {
        'messages': state['messages'] + [feedback],
        'max_f1s': new_max,
        'best_tool_calls': new_best,
    }


# ── Conditional edges ──────────────────────────────────────────────────────────
def tools_condition_edge(state):
    ai_msg = state['messages'][-1]
    if hasattr(ai_msg, 'tool_calls') and ai_msg.tool_calls:
        return 'tools'
    return 'evaluate_node'


def feedback_condition_edge(state):
    return 'llm_node' if state['i'] < n else END


# ── Build graph ─────────────────────────────────────────────────────────────────
builder = StateGraph(State)
builder.add_node('llm_node', llm_node)
builder.add_node('tools', ToolNode(tools))
builder.add_node('evaluate_node', evaluate_node)

builder.add_edge(START, 'llm_node')
builder.add_conditional_edges('llm_node', tools_condition_edge, ['tools', 'evaluate_node'])
builder.add_edge('tools', 'llm_node')
builder.add_conditional_edges('evaluate_node', feedback_condition_edge, ['llm_node', END])

graph = builder.compile(checkpointer=MemorySaver())
display(Image(graph.get_graph().draw_mermaid_png()))
print('Graph compiled.')

In [ ]:
################################################################################
# Cell 6 — Run pipeline (primary seed)
################################################################################

system_message = SystemMessage(
    f'You are a skilled security data analyst.\n'
    f'You are provided with network data entries categorized as either normal or attack, '
    f'along with their corresponding feature names.\n'
    f'Carefully analyze the differences between normal and attack entries by comparing corresponding fields.\n'
    f'Your task is to generate exactly {k} simple and deterministic rules for the top {k} important features '
    f'to filter attack entries.\n'
    f'Supported operators: >, <, >=, <=, ==, !=\n'
    f'Generate exactly {k} rules and make a tool call for each rule.'
)

human_message = HumanMessage(
    f'Analyze the following network data and generate {k} rules to identify attack entries.\n\n'
    f'Normal Entries:\n```{normal_entries}```\n\n'
    f'Attack Entries:\n```{attack_entries}```'
)

initial_state = State(
    i=0,
    max_f1s=0.5,
    best_tool_calls=[],
    messages=[system_message, human_message]
)
config = {'configurable': {'thread_id': f'zd-{WITHHELD_CLASS}-seed{SEED_MAIN}'}}

print(f'Running pipeline — withheld class: {WITHHELD_CLASS}, seed: {SEED_MAIN}')
print(f'Training on {len(normal_df_train):,} normal + {len(attack_df_train):,} known-attack samples')
print(f'Zero-day test set: {len(zeroday_df_test):,} samples of "{WITHHELD_CLASS}"\n')

output = graph.invoke(initial_state, config)

best_tool_calls = output['best_tool_calls']
best_f1 = output['max_f1s']
print(f'\nBest known-attack F1: {best_f1:.4f}')
print(f'Rules ({len(best_tool_calls)}):')
for tc in best_tool_calls:
    a = tc['args']
    print(f'  {a["feature_name"]} {a["op"]} {a["value"]}')

In [ ]:
################################################################################
# Cell 7 — Standard evaluation on known attacks (sanity check)
################################################################################

def apply_rules(tool_calls, df_dict: dict) -> dict:
    """Apply majority-vote rules to each class in df_dict. Returns classification report."""
    y_pred, y_true = [], []
    for label, dataset in df_dict.items():
        for i in tqdm(range(len(dataset)), disable=False, ncols=100, desc=f'Eval {label}...'):
            try:
                votes = [
                    'attack' if operators[tc['args']['op']](
                        dataset.iloc[i][tc['args']['feature_name']], float(tc['args']['value'])
                    ) else 'normal'
                    for tc in tool_calls
                ]
                y_pred.append(mode(votes))
            except Exception:
                y_pred.append('normal')
            y_true.append(label)
    return classification_report(y_true, y_pred, digits=4, output_dict=True), \
           confusion_matrix(y_true, y_pred, labels=['normal', 'attack'])


print('=== Standard Binary Evaluation (known attacks only) ===')
known_report, known_matrix = apply_rules(
    best_tool_calls,
    {'normal': normal_df_test, 'attack': attack_df_test}
)
print(f'Macro F1 : {known_report["macro avg"]["f1-score"]:.4f}')
print(f'Precision: {known_report["macro avg"]["precision"]:.4f}')
print(f'Recall   : {known_report["macro avg"]["recall"]:.4f}')
print(f'\nConfusion matrix (rows=true, cols=pred, order=[normal, attack]):')
print(known_matrix)

In [ ]:
################################################################################
# Cell 8 — Zero-day evaluation (the novel test)
################################################################################


def evaluate_zero_day(tool_calls, zd_df: pd.DataFrame) -> tuple:
    """
    Apply rules to the withheld class.
    Returns (ZDR, list of per-entry predictions).
    ZDR = fraction of entries classified as 'attack'.
    """
    preds = []
    for i in tqdm(range(len(zd_df)), ncols=100, desc=f'Zero-day eval ({WITHHELD_CLASS})...'):
        try:
            votes = [
                'attack' if operators[tc['args']['op']](
                    zd_df.iloc[i][tc['args']['feature_name']], float(tc['args']['value'])
                ) else 'normal'
                for tc in tool_calls
            ]
            preds.append(mode(votes))
        except Exception:
            preds.append('normal')

    zdr = sum(p == 'attack' for p in preds) / len(preds)
    return zdr, preds


llm_zdr, llm_preds = evaluate_zero_day(best_tool_calls, zeroday_df_test)

print(f'\n=== Zero-Day Evaluation: {WITHHELD_CLASS} ===' )
print(f'Total zero-day samples:  {len(zeroday_df_test):,}')
print(f'Detected as attack:      {sum(p == "attack" for p in llm_preds):,}')
print(f'Detected as normal:      {sum(p == "normal" for p in llm_preds):,}')
print(f'\nZero-Day Detection Rate (ZDR): {llm_zdr:.4f}  ({llm_zdr*100:.1f}%)')

zdr_label = 'Strong' if llm_zdr > 0.8 else ('Moderate' if llm_zdr > 0.5 else 'Weak')
print(f'Generalisation:              {zdr_label}')

In [ ]:
################################################################################
# Cell 9 — ML baseline comparison (DT and RF)
################################################################################

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

X_train = train_df[feature_cols].values
y_train = train_df['label'].values
X_test  = test_known_df[feature_cols].values
y_test  = test_known_df['label'].values
X_zd    = zeroday_df_test.values

results_summary = []

for name, model in [
    ('Decision Tree', DecisionTreeClassifier(random_state=SEED_MAIN)),
    ('Random Forest', RandomForestClassifier(n_estimators=100, random_state=SEED_MAIN, n_jobs=-1)),
]:
    model.fit(X_train, y_train)

    # Standard eval
    y_pred_test = model.predict(X_test)
    rep = classification_report(y_test, y_pred_test, output_dict=True)
    known_f1 = rep['macro avg']['f1-score']

    # Zero-day eval
    y_pred_zd = model.predict(X_zd)
    ml_zdr = (y_pred_zd == 'attack').mean()

    results_summary.append([name, f'{known_f1:.4f}', f'{ml_zdr:.4f}  ({ml_zdr*100:.1f}%)'])
    print(f'{name}: known-attack F1={known_f1:.4f}, ZDR={ml_zdr:.4f}')

# Add LLM result
results_summary.insert(0, [
    'LLM-generated rules',
    f'{known_report["macro avg"]["f1-score"]:.4f}',
    f'{llm_zdr:.4f}  ({llm_zdr*100:.1f}%)'
])

In [ ]:
################################################################################
# Cell 10 — Results summary table
################################################################################

print(f'\n=== ZERO-DAY EXPERIMENT RESULTS ===')
print(f'Dataset:      CICIoT2023')
print(f'Withheld:     {WITHHELD_CLASS}')
print(f'Seed:         {SEED_MAIN}')
print()
print(tabulate(
    results_summary,
    headers=['Method', 'Known-Attack F1', f'ZDR ({WITHHELD_CLASS})'],
    tablefmt='grid'
))

print(f'\nInterpretation:')
print(f'  ZDR > 80%: Strong generalisation (rules capture abstract attack indicators)')
print(f'  ZDR 50–80%: Moderate generalisation')
print(f'  ZDR < 50%: Weak generalisation (rules overfit to known attack profiles)')

print(f'\nNote: The LLM rules were generated with NO knowledge of {WITHHELD_CLASS}.')
print(f'      Any detection is pure generalisation from the {known_atk_df["label"].nunique()} known attack classes.')

In [ ]:
################################################################################
# Cell 11 — Repeat with other withheld classes (Round 2, 3)
#
# To run a different withheld class:
#   1. Change WITHHELD_CLASS in Cell 0 to one of:
#      'SqlInjection', 'MITM-ArpSpoofing', 'XSS', 'CommandInjection'
#   2. Restart kernel and run all cells
#
# This cell collects results across multiple runs (requires running each separately).
################################################################################

# Accumulate results here after each run (manual collection)
all_results = {
    # 'Backdoor_Malware': {'LLM': zdr, 'DT': dt_zdr, 'RF': rf_zdr},
    # 'SqlInjection':     {'LLM': zdr, 'DT': dt_zdr, 'RF': rf_zdr},
    # 'MITM-ArpSpoofing': {'LLM': zdr, 'DT': dt_zdr, 'RF': rf_zdr},
}

# Uncomment and populate after running all three classes:
# print(tabulate(
#     [[cls, v['LLM'], v['DT'], v['RF']] for cls, v in all_results.items()],
#     headers=['Withheld Class', 'LLM ZDR', 'DT ZDR', 'RF ZDR'],
#     tablefmt='grid'
# ))

print('Run Cells 0-10 with different WITHHELD_CLASS values to populate all_results above.')
print(f'Current run: WITHHELD_CLASS = "{WITHHELD_CLASS}"')

In [ ]:
################################################################################
# Cell 12 — Visualise rules and zero-day feature coverage
################################################################################

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# What features did the rules use?
rule_features = [tc['args']['feature_name'] for tc in best_tool_calls]
rule_ops      = [tc['args']['op'] for tc in best_tool_calls]
rule_values   = [tc['args']['value'] for tc in best_tool_calls]

# For each rule feature, compare mean value between zero-day class and known attacks
zd_means     = zeroday_df_test[rule_features].mean()
known_means  = attack_df_train[rule_features].mean()
benign_means = normal_df_train[rule_features].mean()

fig, axes = plt.subplots(1, len(rule_features), figsize=(4 * len(rule_features), 4))
if len(rule_features) == 1:
    axes = [axes]

for ax, feat, op_str, thresh in zip(axes, rule_features, rule_ops, rule_values):
    bars = ax.bar(
        ['Benign', 'Known\nAttacks', f'Zero-day\n({WITHHELD_CLASS[:12]})'],
        [benign_means[feat], known_means[feat], zd_means[feat]],
        color=['#2196F3', '#F44336', '#FF9800']
    )
    ax.axhline(y=float(thresh), color='black', linestyle='--', linewidth=1.5,
               label=f'Threshold: {op_str} {thresh}')
    ax.set_title(f'{feat}', fontsize=9, wrap=True)
    ax.set_ylabel('Mean value')
    ax.legend(fontsize=7)

fig.suptitle(
    f'Rule Features: Mean Values vs. Threshold\n'
    f'Withheld zero-day class: {WITHHELD_CLASS}',
    fontsize=11
)
plt.tight_layout()
plt.savefig(f'results/zeroday-feature-means-{WITHHELD_CLASS}.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')